In [1]:

# %%
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Configuración de semillas para reproducibilidad
np.random.seed(42); tf.random.set_seed(42)

# Carga y preparación de datos (Dataset de Cáncer de Mama)
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# %% [markdown]
# ## 1. Modelo Base (Sin Regularización)
# Este modelo es propenso al overfitting debido a su alta capacidad y falta de restricciones.

# %%
def crear_modelo_base():
    model = keras.Sequential([
        layers.Input(shape=(X_train_s.shape[1],)),
        layers.Dense(256, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

modelo_base = crear_modelo_base()
historia_base = modelo_base.fit(X_train_s, y_train, validation_split=0.2, epochs=100, verbose=0)

# %% [markdown]
# ## 2. Modelo con Regularización (L2 + Dropout + Early Stopping)
# Aplicamos técnicas para controlar la varianza y mejorar la generalización.

# %%
def crear_modelo_reg():
    model = keras.Sequential([
        layers.Input(shape=(X_train_s.shape[1],)),
        # Regularización L2 en los pesos
        layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(0.01)),
        layers.Dropout(0.4), # Dropout para apagar neuronas aleatoriamente
        layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.01)),
        layers.Dropout(0.4),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

# Callback de Early Stopping para detener el entrenamiento si la val_loss no mejora
es = callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

modelo_reg = crear_modelo_reg()
historia_reg = modelo_reg.fit(X_train_s, y_train, validation_split=0.2, epochs=100, callbacks=[es], verbose=0)

# %% [markdown]
# ## 3. Análisis de Resultados
# Comparación de métricas entre entrenamiento y validación.

# %%
def graficar_historial(h_base, h_reg):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(h_base.history['loss'], label='Base Train')
    plt.plot(h_base.history['val_loss'], label='Base Val', linestyle='--')
    plt.title('Modelo Base: Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(h_reg.history['loss'], label='Reg Train')
    plt.plot(h_reg.history['val_loss'], label='Reg Val', linestyle='--')
    plt.title('Modelo Regularizado: Loss')
    plt.legend()
    plt.show()

graficar_historial(historia_base, historia_reg)



ModuleNotFoundError: No module named 'tensorflow'